# 🌾 AgroGuard AI - Model Training on Google Colab

**Steps:**
1. Upload your dataset ZIP to Google Drive
2. Run this notebook with GPU enabled
3. Download the trained model files

**Runtime:** Go to Runtime → Change runtime type → Select **T4 GPU**

## Step 1: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

## Step 2: Install Dependencies

In [ ]:
!pip install tensorflow numpy matplotlib opencv-python-headless pillow scikit-learn -q

## Step 3: Upload Dataset

**Option A:** Upload ZIP file from your computer

In [ ]:
from google.colab import files
import zipfile
import os

# Upload the PlantVillage dataset ZIP
print("📤 Upload your PlantVillage dataset ZIP file...")
uploaded = files.upload()

# Extract the ZIP
for filename in uploaded.keys():
    print(f"\n📦 Extracting {filename}...")
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/archive')
    print("✅ Extraction complete!")

**Option B:** Use dataset from Google Drive (faster if already uploaded)

In [ ]:
# Uncomment and modify this if your dataset is in Google Drive
# !unzip -q '/content/drive/MyDrive/PlantVillage.zip' -d /content/archive

## Step 4: Prepare Dataset (Split into Train/Test)

In [ ]:
import os
import shutil
import random
from pathlib import Path

# Configuration
SOURCE_DIR = "/content/archive/PlantVillage"
TRAIN_DIR = "/content/dataset/train"
TEST_DIR = "/content/dataset/test"

SELECTED_CLASSES = {
    "Pepper__bell___Bacterial_spot": "Pepper_bell_Bacterial_spot",
    "Pepper__bell___healthy": "Pepper_bell_healthy",
    "Potato___Early_blight": "Potato_Early_blight",
    "Potato___Late_blight": "Potato_Late_blight",
    "Potato___healthy": "Potato_healthy",
    "Tomato_Early_blight": "Tomato_Early_blight",
    "Tomato_Late_blight": "Tomato_Late_blight",
    "Tomato_healthy": "Tomato_healthy",
    "Tomato_Bacterial_spot": "Tomato_Bacterial_spot",
    "Tomato_Leaf_Mold": "Tomato_Leaf_Mold",
    "Tomato_Septoria_leaf_spot": "Tomato_Septoria_leaf_spot",
    "Tomato__Target_Spot": "Tomato_Target_Spot",
    "Tomato__Tomato_mosaic_virus": "Tomato_Mosaic_virus",
    "Tomato__Tomato_YellowLeaf__Curl_Virus": "Tomato_YellowLeaf_Curl_Virus",
    "Tomato_Spider_mites_Two_spotted_spider_mite": "Tomato_Spider_mites",
}

TRAIN_SPLIT = 0.8
MAX_PER_CLASS = 400  # Reduce for faster training

print("📂 Preparing dataset...\n")

train_path = Path(TRAIN_DIR)
test_path = Path(TEST_DIR)

for source_class, target_class in SELECTED_CLASSES.items():
    source_folder = Path(SOURCE_DIR) / source_class
    
    if not source_folder.exists():
        print(f"  ⚠️  Skipping {source_class} - not found")
        continue
    
    images = list(source_folder.glob("*.JPG")) + list(source_folder.glob("*.jpg")) + list(source_folder.glob("*.png"))
    
    if not images:
        continue
    
    if MAX_PER_CLASS and len(images) > MAX_PER_CLASS:
        images = random.sample(images, MAX_PER_CLASS)
    
    random.shuffle(images)
    split_idx = int(len(images) * TRAIN_SPLIT)
    train_images = images[:split_idx]
    test_images = images[split_idx:]
    
    train_class_dir = train_path / target_class
    test_class_dir = test_path / target_class
    train_class_dir.mkdir(parents=True, exist_ok=True)
    test_class_dir.mkdir(parents=True, exist_ok=True)
    
    for img in train_images:
        shutil.copy2(img, train_class_dir / img.name)
    
    for img in test_images:
        shutil.copy2(img, test_class_dir / img.name)
    
    print(f"  ✅ {target_class}: Train={len(train_images)}, Test={len(test_images)}")

print("\n✅ Dataset preparation complete!")

## Step 5: Build and Train CNN Model

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# Configuration
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 20

os.makedirs("/content/model", exist_ok=True)

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
)

test_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
)

test_data = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
)

NUM_CLASSES = train_data.num_classes
print(f"\n✅ Found {NUM_CLASSES} classes: {list(train_data.class_indices.keys())}\n")

# Save class names
with open("/content/model/class_names.json", "w") as f:
    json.dump(train_data.class_indices, f, indent=2)
print("✅ class_names.json saved.")

In [ ]:
# Build CNN Model
model = models.Sequential([
    # Block 1
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(128, 128, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.2),

    # Block 2
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.2),

    # Block 3
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.2),

    # Block 4
    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D(2, 2),
    layers.Dropout(0.2),

    # Dense layers
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    ModelCheckpoint(
        "/content/model/crop_disease_model.h5",
        save_best_only=True,
        monitor="val_accuracy",
        verbose=1
    ),
    ReduceLROnPlateau(
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        monitor="val_loss"
    ),
]

# Start Training
print("\n🚀 Starting training with GPU acceleration...\n")
history = model.fit(
    train_data,
    validation_data=test_data,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## Step 6: Plot Training Results

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"], label="Train Acc")
axes[0].plot(history.history["val_accuracy"], label="Val Acc")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history["loss"], label="Train Loss")
axes[1].plot(history.history["val_loss"], label="Val Loss")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig("/content/model/training_curves.png", dpi=150)
plt.show()

print("\n✅ Training complete!")
print(f"Final Training Accuracy: {history.history['accuracy'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]:.4f}")

## Step 7: Download Trained Model Files

In [ ]:
# Create a ZIP file with model files
import shutil

shutil.make_archive('/content/trained_model', 'zip', '/content/model')

print("📦 Model files packaged!")
print("\nDownloading model files...")

files.download('/content/trained_model.zip')

print("\n✅ Download complete!")
print("\nExtract the ZIP and copy these files to your backend/model/ folder:")
print("  - crop_disease_model.h5")
print("  - class_names.json")
print("  - training_curves.png")

## Step 8: Test the Model (Optional)

In [ ]:
# Load class names
with open("/content/model/class_names.json") as f:
    class_data = json.load(f)

class_names = [k for k, v in sorted(class_data.items(), key=lambda x: x[1])]

# Test on a sample image
import cv2
from tensorflow.keras.models import load_model

# Get a random test image
test_class = list(Path(TEST_DIR).iterdir())[0]
test_image = list(test_class.glob("*.JPG"))[0]

# Preprocess
img = cv2.imread(str(test_image))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img, (128, 128))
img_array = img_resized / 255.0
img_array = np.reshape(img_array, (1, 128, 128, 3))

# Predict
predictions = model.predict(img_array)[0]
top_idx = np.argmax(predictions)
confidence = predictions[top_idx] * 100
predicted_class = class_names[top_idx]

# Display
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.title(f"Prediction: {predicted_class}\nConfidence: {confidence:.2f}%", fontsize=14)
plt.axis('off')
plt.show()

print(f"\n✅ Predicted: {predicted_class}")
print(f"   Confidence: {confidence:.2f}%")
print(f"   Actual: {test_class.name}")